### Part 2: Word2Vec + Cosine Ranking Score


In [1]:
# Install required packages (quiet mode). Run once per environment.
%pip install -q gensim scikit-learn numpy pandas


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import csv
from collections import defaultdict
from array import array
from typing import Dict, List, Tuple
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords



/Users/adriasoria/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adriasoria/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/adriasoria/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/adriasoria/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
def build_terms_query(text: str) -> List[str]:
    """Tokenize and normalize text into a list of lowercase terms.
    - Keeps only alphanumeric characters
    - Splits on non-alphanumerics
    """
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))
    if not text:
        return []
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return tokens


def build_terms_doc(text: str) -> List[str]:
    """Tokenize text into a list of terms.
    """
    if not text:
        return []
    tokens = word_tokenize(text)
    return tokens

In [5]:
DATA_PATH = "../../data/productos_preprocesados.csv"

# Load CSV
usecols = None  # read all columns
products_df = pd.read_csv(DATA_PATH, usecols=usecols)

terms = []
with open(DATA_PATH, mode="r", encoding="utf-8", newline="") as f:   
    reader = csv.DictReader(f)
    documentos = list(reader)
    num_documents = len(documentos)

    for row in documentos:
        doc_id = row.get("pid")
        if not doc_id:
            # Skip rows without a valid identifier
            continue

        title = (row.get("title") or "").strip()
        description = (row.get("description") or "").strip()
        brand = (row.get("brand") or "").strip()

        # Concatenate selected fields
        content = f"{title} {description} {brand}"

        terms.append(list(build_terms_doc(content)))


In [6]:
terms[0][:20] if len(terms) else []

['solid',
 'women',
 'multicolor',
 'track',
 'pant',
 'yorker',
 'trackpant',
 'made',
 'rich',
 'comb',
 'cotton',
 'give',
 'rich',
 'lookdesign',
 'comfortskin',
 'friendli',
 'fabricitchfre',
 'waistband',
 'great',
 'year']

In [7]:
# Train Word2Vec model on the product corpus
w2v_model = Word2Vec(
    sentences=terms,
    vector_size=200,
    window=10,
    min_count=10,
    workers=4,
    sg=1,
    epochs=20,
)
# From Seminar:
# vector_size=100, window=10, min_count=10, negative=15, sg=0

w2v_dim = w2v_model.wv.vector_size
w2v_dim


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


200

In [8]:
# Compute average Word2Vec vectors for each document
import math

def average_vector(tokens: list[str], model: Word2Vec) -> np.ndarray:
    if not tokens:
        return np.zeros(model.wv.vector_size, dtype=np.float32)
    vectors = []
    for tok in tokens:
        if tok in model.wv:
            vectors.append(model.wv[tok])
    if not vectors:
        return np.zeros(model.wv.vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0)

# Build document matrix
doc_vectors = np.vstack([average_vector(tokens, w2v_model) for tokens in terms])

doc_vectors.shape


(28080, 200)

In [9]:
queries_part2 = [
    "cotton multicolor track pant",
    "women black track pant pockets",
    "men cotton blue pant",
    "side pocket cotton pant",
    "slim cotton black pant"
]

queries_part2


['cotton multicolor track pant',
 'women black track pant pockets',
 'men cotton blue pant',
 'side pocket cotton pant',
 'slim cotton black pant']

In [10]:
from typing import Dict

def search_top20_w2v(query: str, top_n: int = 20) -> pd.DataFrame:
    """
    Finds all documents that contain all query terms (conjunctive/AND semantics),
    then ranks these documents by cosine similarity between the query Word2Vec vector
    and each document's Word2Vec vector. Return top_n with metadata.
    """
    # Build query Word2Vec vector and get query tokens
    q_tokens = build_terms_query(query)
    q_w2v = average_vector(q_tokens, w2v_model)

    # Find documents that contain ALL query tokens
    matching_idx = []
    set_qtokens = set(q_tokens)
    for idx, doc_tokens in enumerate(terms):
        if set_qtokens.issubset(set(doc_tokens)):
            matching_idx.append(idx)

    if not matching_idx:
        # No documents match ALL terms, return empty DataFrame with columns
        return pd.DataFrame(columns=["pid", "title", "url", "cosine", "rank"])

    # Only keep doc_vectors and product metadata for matching docs
    filtered_vectors = doc_vectors[matching_idx]
    filtered_products = products_df.iloc[matching_idx]

    # Compute cosine similarity for matching docs
    cos = cosine_similarity(filtered_vectors, q_w2v.reshape(1, -1)).ravel()

    n_docs = cos.shape[0]
    n = min(top_n, n_docs)
    top_idx = np.argpartition(cos, -n)[-n:]
    top_idx = top_idx[np.argsort(cos[top_idx])[::-1]]  # sort by cosine desc

    out = filtered_products.iloc[top_idx][["pid", "title"]].copy()
    out["url"] = filtered_products.get("url", pd.Series([""] * len(filtered_products))).iloc[top_idx].values
    out["cosine"] = cos[top_idx]
    out["rank"] = np.arange(1, len(out) + 1)
    return out.reset_index(drop=True)

# Convenience: run for a list of queries
def run_queries_top20(queries: list[str]) -> Dict[str, pd.DataFrame]:
    results: Dict[str, pd.DataFrame] = {}
    for q in queries:
        results[q] = search_top20_w2v(q, top_n=20)
    return results


In [11]:
results_by_query = run_queries_top20(queries_part2)

for q in queries_part2:
    print("\n-> Query:", q)
    display(results_by_query.get(q, pd.DataFrame()).head(20))



-> Query: cotton multicolor track pant


,pid,title,url,cosine,rank
0,TKPFDHDHGWXCFXPC,solid men multicolor track pant,https://www.flipkart.com/clothin-solid-men-mul...,0.663464,1
1,TKPFDHDGEYWM99XG,solid women multicolor track pant,https://www.flipkart.com/clothin-solid-men-mul...,0.630276,2
2,TKPFWJF7GG7YQSXM,checker women multicolor track pant,https://www.flipkart.com/u-s-polo-association-...,0.620286,3
3,TKPEZAGS42FZVHYJ,solid women multicolor track pant,https://www.flipkart.com/humbert-solid-men-mul...,0.615057,4
4,TKPEZAGSJSTSUTRZ,solid women multicolor track pant,https://www.flipkart.com/humbert-solid-men-mul...,0.594610,5
5,TKPEZAGRRPZU7GY3,varsiti men multicolor track pant,https://www.flipkart.com/humbert-varsity-men-m...,0.589944,6
6,TKPEZAGS4TSDYFGW,solid women multicolor track pant,https://www.flipkart.com/humbert-solid-men-mul...,0.588366,7
7,TKPEZAGSQNCCZVYS,solid men multicolor track pant,https://www.flipkart.com/humbert-solid-men-mul...,0.586426,8
8,TKPEZAGSUGUYC28P,solid women multicolor track pant,https://www.flipkart.com/humbert-solid-men-mul...,0.579504,9
9,TKPFCZ9EGGYENTZS,color block women multicolor track pant,https://www.flipkart.com/yorker-color-block-me...,0.540383,10



-> Query: women black track pant pockets


,pid,title,url,cosine,rank
0,TKPFTE9A4RNC7HUV,solid women black track pant,https://www.flipkart.com/jack-hardy-solid-men-...,0.808122,1
1,TKPEQX5JXNUQFRFD,solid women black track pant,https://www.flipkart.com/dunamis-solid-men-bla...,0.794928,2
2,TKPEQX5JHTGFMHGC,solid women grey track pant,https://www.flipkart.com/dunamis-solid-men-gre...,0.792359,3
3,TKPFPBJCY6M4UJBU,solid women black track pant,https://www.flipkart.com/u-s-polo-association-...,0.753090,4
4,TKPFYTUEXCNQVK5S,solid women black track pant,https://www.flipkart.com/nettle-solid-men-blac...,0.744133,5
5,TKPFDT6GHGEYDTAS,solid women black track pant,https://www.flipkart.com/reebok-solid-men-blac...,0.722357,6
6,TKPEZAGSCUSJF8VM,solid women green blue track pant,https://www.flipkart.com/humbert-solid-men-gre...,0.713222,7
7,TKPFDHDGS3D3BGUD,solid women black grey track pant,https://www.flipkart.com/clothin-solid-men-bla...,0.701268,8
8,TKPEZAGSJSTSUTRZ,solid women multicolor track pant,https://www.flipkart.com/humbert-solid-men-mul...,0.685788,9
9,TKPEGZ7HBHDKUXTV,solid women black track pant,https://www.flipkart.com/humbert-solid-men-bla...,0.685112,10



-> Query: men cotton blue pant


,pid,title,url,cosine,rank
0,TKPEGZ7HNZJCAYHX,solid men dark blue track pant,https://www.flipkart.com/humbert-solid-men-dar...,0.681756,1
1,TKPEGZ7MRE2MHY6Z,solid men dark blue track pant,https://www.flipkart.com/humbert-solid-men-dar...,0.681157,2
2,TKPEXZA6CUYUWSVS,solid men dark blue track pant,https://www.flipkart.com/humbert-solid-men-dar...,0.661513,3
3,TKPEGZ7HNGCTZRNE,solid men dark blue track pant,https://www.flipkart.com/humbert-solid-men-dar...,0.654368,4
4,TKPFJQFWJB2SPEPP,solid men blue track pant,https://www.flipkart.com/solid-men-blue-track-...,0.643829,5
5,TKPFGFTHHB9NYC9Y,print men dark blue blue track pant,https://www.flipkart.com/zippy-printed-men-dar...,0.637370,6
6,TKPFGFTMV2P2MEEM,print men dark blue grey track pant,https://www.flipkart.com/zippy-printed-men-dar...,0.634786,7
7,TKPFGFTMRXWJJM4V,print men dark blue grey track pant,https://www.flipkart.com/zippy-printed-men-dar...,0.634786,8
8,TKPFYURWE3GGHVDT,solid men blue track pant,https://www.flipkart.com/nettle-solid-men-blue...,0.632924,9
9,TKPFGFTMBZTFPF4D,print men dark blue green track pant,https://www.flipkart.com/zippy-printed-men-dar...,0.631735,10



-> Query: side pocket cotton pant


,pid,title,url,cosine,rank
0,TKPFW6455HJV2JVH,solid men maroon track pant,https://www.flipkart.com/u-s-polo-association-...,0.751214,1
1,TKPFDHDHYACU53ZG,solid men black track pant,https://www.flipkart.com/clothin-solid-men-bla...,0.751119,2
2,TKPFDHDGS3D3BGUD,solid women black grey track pant,https://www.flipkart.com/clothin-solid-men-bla...,0.749661,3
3,TKPFDHDGEYWM99XG,solid women multicolor track pant,https://www.flipkart.com/clothin-solid-men-mul...,0.746617,4
4,TKPFZ3MFMFJZHYHP,solid men blue track pant,https://www.flipkart.com/nettle-solid-men-blue...,0.694455,5
5,TKPFZ3HMUTG8ZMZX,solid men black track pant,https://www.flipkart.com/nettle-solid-men-blac...,0.693743,6
6,TKPFZ3ZGSHDZZVHZ,light melang solid men grey track pant,https://www.flipkart.com/nettle-light-melange-...,0.692636,7
7,TKPFZ3T8EPYZFQBT,lava grey solid women grey track pant,https://www.flipkart.com/nettle-lava-grey-soli...,0.692163,8
8,TKPFGK5CUGWRDPN9,self design men grey track pant,https://www.flipkart.com/fleximaa-self-design-...,0.686073,9
9,TKPFGK5CCZ5RARHZ,self design men grey track pant,https://www.flipkart.com/fleximaa-self-design-...,0.686073,10



-> Query: slim cotton black pant


,pid,title,url,cosine,rank
0,TKPFK3W6NVCGUTHR,solid women black track pant,https://www.flipkart.com/reebok-classics-solid...,0.702821,1
1,TROE3KVQD8RAEPUZ,pack black brown slim fit women brown black co...,https://www.flipkart.com/inspire-pack-black-br...,0.688619,2
2,TROE3KVQYZHZG2DV,pack black dkhaki slim fit men brown black cot...,https://www.flipkart.com/inspire-pack-black-d-...,0.682363,3
3,TROEDVF5YUXUH2ZU,slim fit men brown black cotton blend trouser,https://www.flipkart.com/inspire-slim-fit-men-...,0.667984,4
4,TROE367EEBB6BNTC,pack slim fit men blue brown black cotton visc...,https://www.flipkart.com/inspire-pack-3-slim-f...,0.665725,5
5,TROFFXABA4HTGUWE,slim fit women black cotton lycra blend trouser,https://www.flipkart.com/inspire-slim-fit-men-...,0.665708,6
6,TROEE8SEEBTSTA86,slim fit women blue black grey cotton viscos b...,https://www.flipkart.com/inspire-slim-fit-men-...,0.663589,7
7,TROEE9CQFEMEQECZ,slim fit women brown black grey cotton viscos ...,https://www.flipkart.com/inspire-slim-fit-men-...,0.661235,8
8,TROE237TVFDFV5AD,slim fit men black cotton blend trouser,https://www.flipkart.com/inspire-slim-fit-men-...,0.659413,9
9,TROE2HZRCHVZ4YDY,slim fit men black cotton viscos blend trouser,https://www.flipkart.com/inspire-slim-fit-men-...,0.646712,10
